In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import gc
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter

Mounted at /content/drive


In [ ]:
FASTA = "/content/drive/MyDrive/salmonella-enterica.reads.fna"

print("FASTA file:", FASTA)
print("File exists:", os.path.exists(FASTA))

if not os.path.exists(FASTA):
    raise FileNotFoundError(
        "FASTA file not found. Check the path."
    )

FASTA file: /content/drive/MyDrive/salmonella-enterica.reads.fna
File exists: True


In [ ]:
def read_fasta(path):

    sequence = ""

    with open(path, "r") as file:

        for line in file:

            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):

                if sequence:
                    yield sequence.upper()

                sequence = ""

            else:

                sequence += line

        if sequence:
            yield sequence.upper()

abbiamo caricato  il dataset e ora carichiamo e guardiamo i parametri che ci servono

English: We have uploaded the dataset and now we will load and look at the parameters we need.

In [ ]:
reads = list(read_fasta(FASTA))

N = len(reads)

read_lengths = np.array([
    len(read)
    for read in reads
])

l = read_lengths.mean()

total_bases = read_lengths.sum()

L = 5_000_000

ERROR_RATE = 0.01

C = total_bases / L

print("Number of reads:", N)
print("Mean read length:", l)
print("Total sequenced bases:", total_bases)
print("Genome length:", L)
print("Estimated coverage:", round(C, 2))
print("Sequencing error rate:", ERROR_RATE)

Number of reads: 1993167
Mean read length: 250.0
Total sequenced bases: 498291750
Genome length: 5000000
Estimated coverage: 99.66
Sequencing error rate: 0.01


## Sequencing errors and expected k-mer multiplicity

Although theory cannot determine the optimal value of $k$, it is useful to understand how sequencing errors affect the support of true genomic k-mers.

For a specific genomic k-mer, a read of length $l$ can cover it from approximately $l-k+1$ valid starting positions. Therefore, the probability that a given read covers that k-mer is approximately $p_k=(l-k+1)/L$.

With $N$ independent reads, the number of reads covering the k-mer can be modeled as

$$
X_k \sim \operatorname{Binomial}(N,p_k).
$$

If the per-base sequencing error rate is $\varepsilon$, a k-mer is read without errors only if all its $k$ nucleotides are correct. This occurs with probability $(1-\varepsilon)^k$. Therefore, the number of correct observations of a true genomic k-mer can be modeled as

$$
X_k^{\mathrm{true}}
\sim
\operatorname{Binomial}
\left(
N,
p_k(1-\varepsilon)^k
\right).
$$

Since $N$ is large and the probability associated with one particular genomic k-mer is small, the Binomial distribution can be approximated by a Poisson distribution,

$$
X_k^{\mathrm{true}}
\approx
\operatorname{Poisson}
\left(
\lambda_{\mathrm{true}}(k)
\right),
$$

where

$$
\lambda_{\mathrm{true}}(k)
=
C
\frac{l-k+1}{l}
(1-\varepsilon)^k.
$$

Increasing $k$ has two opposite effects. Longer k-mers are more specific and therefore help distinguish repeated genomic regions, but they are also more likely to contain at least one sequencing error. This probability is

$$
P(\text{at least one error})
=
1-(1-\varepsilon)^k.
$$

We compute these quantities for different candidate values of $k$ using the parameters estimated from the large dataset. They are used to interpret the empirical k-mer spectra, but not to determine the optimal $k$ theoretically.


In [ ]:
# Candidate values of k
k_values_theory = [
    15,
    23,
    31,
    39,
    47,
    55
]

theoretical_results = []

for k in k_values_theory:

    # Probability that a read covers a specific genomic k-mer
    p_cover = (
        (l - k + 1) / L
    )

    # Probability that all k bases are sequenced correctly
    p_error_free = (
        (1 - ERROR_RATE) ** k
    )

    # Probability that the k-mer contains at least one error
    p_at_least_one_error = (
        1 - p_error_free
    )

    # Probability that a read contains a correct copy
    # of this specific genomic k-mer
    p_true = (
        p_cover
        * p_error_free
    )

    # Binomial mean N * p_true
    lambda_binomial = (
        N * p_true
    )

    # Equivalent expression using coverage
    lambda_true = (
        C
        * ((l - k + 1) / l)
        * ((1 - ERROR_RATE) ** k)
    )

    theoretical_results.append({
        "k": k,
        "P_error_free": p_error_free,
        "P_at_least_one_error": p_at_least_one_error,
        "lambda_true": lambda_true
    })


theoretical_df = pd.DataFrame(
    theoretical_results
)

display(
    theoretical_df.round(4)
)

,k,P_error_free,P_at_least_one_error,lambda_true
0,15,0.8601,0.1399,80.9121
1,23,0.7936,0.2064,72.1303
2,31,0.7323,0.2677,64.2225
3,39,0.6757,0.3243,57.1061
4,47,0.6235,0.3765,50.7058
5,55,0.5754,0.4246,44.9537


## A theoretical lower bound for $k$

Theory is used only to exclude values of $k$ that are clearly too small. There are at most $4^k$ different DNA sequences of length $k$. If $4^k < L$, there are fewer possible k-mers than genomic positions, so repeated k-mers are unavoidable.

A basic lower bound is therefore $k \geq \log_4(L)\$.

For a genome of approximately $L = 5 \times 10^6\$, we obtain

$$
k \geq \log_4(5 \times 10^6) \approx 11.13,
$$

so, since \(k\) must be an integer,

$$
\boxed{k_{\min}=12}.
$$

This lower bound does not determine the optimal value of \(k\). The final choice will be based on the empirical k-mer spectra.

In [ ]:
k_lower_bound = math.ceil(
    math.log(L, 4)
)

print(
    "Theoretical lower bound:",
    k_lower_bound
)

print(
    "L / 4^k:",
    L / (4 ** k_lower_bound)
)

Theoretical lower bound: 12
L / 4^k: 0.2980232238769531


## Counting canonical k-mers

This function counts how many times each k-mer appears in the reads.

To make the computation faster and more memory-efficient, each nucleotide is encoded using 2 bits:

$A \rightarrow 00, C \rightarrow 01, G \rightarrow 10, T \rightarrow 11$.

For each read, the function scans the sequence one base at a time and updates both the k-mer in the forward direction and its reverse complement.

A k-mer and its reverse complement represent the same genomic sequence on opposite DNA strands, so they are merged using the canonical representation

$$
\mathrm{canonical}(w)=\min\left(w,\mathrm{revcomp}(w)\right).
$$

The final object `counts` stores, for each distinct canonical k-mer, the total number of times it appears in all reads.

These counts are then used to build the k-mer frequency spectrum.

In [ ]:

BASE_TO_BITS = {
    "A": 0,
    "C": 1,
    "G": 2,
    "T": 3
}


def count_canonical_kmers(reads, k):

    counts = Counter()

    mask = (1 << (2 * k)) - 1
    shift = 2 * (k - 1)

    for read in reads:

        forward = 0
        reverse = 0
        valid_length = 0

        for base in read:

            b = BASE_TO_BITS[base]

            # Forward k-mer
            forward = (
                (forward << 2) | b
            ) & mask

            # Reverse-complement k-mer
            complement = 3 - b

            reverse = (
                (reverse >> 2)
                |
                (complement << shift)
            )

            valid_length += 1

            if valid_length >= k:

                canonical = min(
                    forward,
                    reverse
                )

                counts[canonical] += 1

    return counts

In [ ]:
def build_kmer_spectrum(counts):

    return Counter(
        counts.values()
    )

## Empirical interpretation of the k-mer spectrum

For a fixed value of $k$, the k-mer spectrum contains different components.

Low-multiplicity k-mers are mainly associated with sequencing errors.

The main genomic peak corresponds approximately to solid k-mers occurring once
in the genome.

K-mers occurring in repeated genomic regions tend to have higher multiplicity.
For example, a sequence present twice in the genome is expected to have roughly
twice the multiplicity of a single-copy sequence.

We therefore estimate, for each candidate $k$:

1. the fraction of distinct k-mers likely to be erroneous;
2. the fraction of solid k-mers likely to occur more than once in the genome.

The selected $k$ should provide a reasonable compromise between these two
quantities.

In [ ]:
# ============================================================
# CANDIDATE k VALUES
# ============================================================

K_VALUES = [
    15,
    23,
    31,
    39,
    47,
    55
]

print("Candidate k values:", K_VALUES)

Candidate k values: [15, 23, 31, 39, 47, 55]


In [ ]:
def smooth_signal(values, window=7):

    values = np.asarray(
        values,
        dtype=float
    )

    if len(values) < window:
        return values

    if window % 2 == 0:
        window += 1

    kernel = (
        np.ones(window)
        / window
    )

    return np.convolve(
        values,
        kernel,
        mode="same"
    )


def find_peak_and_threshold(
    spectrum,
    coverage,
    window=7
):

    # --------------------------------------------------------
    # Search range
    #
    # The dataset has high coverage (~100x).
    # Very low multiplicities are dominated by errors.
    # --------------------------------------------------------

    max_observed = max(
        spectrum.keys()
    )

    max_m = min(
        max_observed,
        int(1.5 * coverage)
    )

    x = np.arange(
        1,
        max_m + 1
    )

    y = np.array([
        spectrum.get(m, 0)
        for m in x
    ], dtype=float)

    log_y = np.log1p(y)

    smooth_log = smooth_signal(
        log_y,
        window=window
    )

    # --------------------------------------------------------
    # Empirical genomic peak
    #
    # We exclude the very low multiplicity error component.
    # Coverage is used only to define a broad search region,
    # not to choose k.
    # --------------------------------------------------------

    peak_search_min = max(
        5,
        int(0.10 * coverage)
    )

    peak_mask = (
        x >= peak_search_min
    )

    peak_x = x[
        peak_mask
    ]

    peak_y = smooth_log[
        peak_mask
    ]

    genomic_peak = int(
        peak_x[
            np.argmax(peak_y)
        ]
    )

    # --------------------------------------------------------
    # Valley between error component and genomic component
    # --------------------------------------------------------

    valley_mask = (
        (x >= 2)
        &
        (x < genomic_peak)
    )

    valley_x = x[
        valley_mask
    ]

    valley_y = smooth_log[
        valley_mask
    ]

    minimum = valley_y.min()

    genomic_level = smooth_log[
        genomic_peak - 1
    ]

    # Instead of choosing one noisy minimum,
    # identify the bottom region of the valley.
    valley_level = (
        minimum
        +
        0.15
        * (genomic_level - minimum)
    )

    low_region = (
        valley_y <= valley_level
    )

    candidate_x = valley_x[
        low_region
    ]

    if len(candidate_x) == 0:

        threshold = int(
            valley_x[
                np.argmin(valley_y)
            ]
        )

    else:

        # Central point of the broad valley
        threshold = int(
            np.median(candidate_x)
        )

    return genomic_peak, threshold

In [ ]:


def empirical_kmer_statistics(
    spectrum,
    genomic_peak,
    threshold
):

    # --------------------------------------------------------
    # Total number of distinct k-mer types
    # --------------------------------------------------------

    total_types = sum(
        spectrum.values()
    )

    # Total number of observed k-mer occurrences
    total_occurrences = sum(
        multiplicity * n_types
        for multiplicity, n_types
        in spectrum.items()
    )

    # --------------------------------------------------------
    # Likely erroneous k-mers
    #
    # Multiplicity below the valley threshold
    # --------------------------------------------------------

    erroneous_types = sum(
        n_types
        for multiplicity, n_types
        in spectrum.items()
        if multiplicity < threshold
    )

    erroneous_occurrences = sum(
        multiplicity * n_types
        for multiplicity, n_types
        in spectrum.items()
        if multiplicity < threshold
    )

    erroneous_type_fraction = (
        erroneous_types
        / total_types
    )

    erroneous_occurrence_fraction = (
        erroneous_occurrences
        / total_occurrences
    )

    # --------------------------------------------------------
    # Solid k-mers
    # --------------------------------------------------------

    solid_types = (
        total_types
        - erroneous_types
    )

    # --------------------------------------------------------
    # Likely repeated genomic k-mers
    #
    # A single-copy k-mer is centered around genomic_peak.
    # A two-copy k-mer is expected around 2 * genomic_peak.
    #
    # The midpoint 1.5 * genomic_peak is therefore used
    # as an empirical boundary.
    # --------------------------------------------------------

    repeat_threshold = int(
        np.ceil(
            1.5 * genomic_peak
        )
    )

    repeated_types = sum(
        n_types
        for multiplicity, n_types
        in spectrum.items()
        if multiplicity >= repeat_threshold
    )

    if solid_types > 0:

        repeated_fraction_among_solid = (
            repeated_types
            / solid_types
        )

    else:

        repeated_fraction_among_solid = np.nan

    return {
        "total_distinct_kmers":
            total_types,

        "erroneous_type_fraction":
            erroneous_type_fraction,

        "erroneous_occurrence_fraction":
            erroneous_occurrence_fraction,

        "solid_kmers":
            solid_types,

        "repeat_threshold":
            repeat_threshold,

        "repeated_fraction_among_solid":
            repeated_fraction_among_solid
    }

In [ ]:
# ============================================================
# EMPIRICAL ANALYSIS ACROSS k
# ============================================================

results = []

spectra = {}


for k in K_VALUES:

    print()
    print("=" * 55)
    print("Processing k =", k)
    print("=" * 55)

    # --------------------------------------------------------
    # Count canonical k-mers
    # --------------------------------------------------------

    counts = count_canonical_kmers(
        reads,
        k
    )

    spectrum = build_kmer_spectrum(
        counts
    )

    # Spectrum is small enough to retain
    spectra[k] = spectrum.copy()

    # --------------------------------------------------------
    # Empirical genomic peak and filtering threshold
    # --------------------------------------------------------

    genomic_peak, threshold = (
        find_peak_and_threshold(
            spectrum,
            coverage=C
        )
    )

    # --------------------------------------------------------
    # Empirical fractions
    # --------------------------------------------------------

    stats = empirical_kmer_statistics(
        spectrum,
        genomic_peak,
        threshold
    )

    # --------------------------------------------------------
    # Theoretical sequencing-error probability
    #
    # This is reported only for interpretation.
    # It is NOT used to select k.
    # --------------------------------------------------------

    p_error_theoretical = (
        1
        -
        (1 - ERROR_RATE) ** k
    )

    results.append({

        "k":
            k,

        "genomic_peak":
            genomic_peak,

        "threshold":
            threshold,

        "repeat_threshold":
            stats["repeat_threshold"],

        "distinct_kmers":
            stats["total_distinct_kmers"],

        "erroneous_type_fraction":
            stats[
                "erroneous_type_fraction"
            ],

        "erroneous_occurrence_fraction":
            stats[
                "erroneous_occurrence_fraction"
            ],

        "repeated_fraction_among_solid":
            stats[
                "repeated_fraction_among_solid"
            ],

        "theoretical_P_error":
            p_error_theoretical
    })

    print(
        "Genomic peak:",
        genomic_peak
    )

    print(
        "Valley threshold:",
        threshold
    )

    print(
        "Repeat threshold:",
        stats["repeat_threshold"]
    )

    print(
        "Likely erroneous k-mer fraction:",
        round(
            stats[
                "erroneous_type_fraction"
            ],
            4
        )
    )

    print(
        "Likely repeated fraction among solid k-mers:",
        round(
            stats[
                "repeated_fraction_among_solid"
            ],
            4
        )
    )

    # --------------------------------------------------------
    # Free the very large k-mer dictionary
    # --------------------------------------------------------

    del counts
    del spectrum

    gc.collect()


results_df = pd.DataFrame(
    results
)

display(
    results_df.round(6)
)


Processing k = 15
Genomic peak: 90
Valley threshold: 31
Repeat threshold: 135
Likely erroneous k-mer fraction: 0.7777
Likely repeated fraction among solid k-mers: 0.0284

Processing k = 23


KeyboardInterrupt: 

In [ ]:
!apt-get update -qq
!apt-get install -y jellyfish

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libhts3t64 libhtscodecs2 libjellyfish-2.0-2
The following NEW packages will be installed:
  jellyfish libhts3t64 libhtscodecs2 libjellyfish-2.0-2
0 upgraded, 4 newly installed, 0 to remove and 24 not upgraded.
Need to get 1,362 kB of archives.
After this operation, 3,703 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 libhtscodecs2 amd64 1.6.0-1build1 [98.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 libhts3t64 amd64 1.19+ds-1.1build3 [437 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/universe amd64 libjellyfish-2.0-2 amd64 2.3.1-3build1 [68.8 kB]
Get:4 http://archive.ubuntu.

In [ ]:
# Original file on Google Drive
FASTA_DRIVE = "/content/drive/MyDrive/salmonella-enterica.reads.fna"

# Local copy used for the computations
FASTA = "/content/salmonella-enterica.reads.fna"

!cp "$FASTA_DRIVE" "$FASTA"

print("FASTA used for the analysis:", FASTA)
print("File exists:", os.path.exists(FASTA))

FASTA used for the analysis: /content/salmonella-enterica.reads.fna
File exists: True


In [ ]:
def count_kmer_spectrum_jellyfish(fasta_path, k):

    jf_file = f"/content/kmers_k{k}.jf"
    histo_file = f"/content/kmers_k{k}_histo.txt"

    threads = os.cpu_count() or 2

    print(
        f"Counting canonical k-mers for k = {k} "
        f"using {threads} threads..."
    )

    # --------------------------------------------------------
    # Count canonical k-mers
    # --------------------------------------------------------

    subprocess.run([
        "jellyfish",
        "count",
        "-m", str(k),
        "-s", "100M",
        "-t", str(threads),
        "-C",
        "-o", jf_file,
        fasta_path
    ], check=True)

    # --------------------------------------------------------
    # Build the multiplicity histogram
    # --------------------------------------------------------

    with open(histo_file, "w") as out:

        subprocess.run([
            "jellyfish",
            "histo",
            "-t", str(threads),
            jf_file
        ],
        stdout=out,
        check=True)

    # --------------------------------------------------------
    # Convert Jellyfish histogram into a Python Counter
    #
    # spectrum[m] =
    # number of distinct k-mers observed exactly m times
    # --------------------------------------------------------

    spectrum = Counter()

    with open(histo_file, "r") as file:

        for line in file:

            multiplicity, number_of_kmers = map(
                int,
                line.split()
            )

            spectrum[multiplicity] = number_of_kmers

    # Temporary files are no longer needed
    os.remove(jf_file)
    os.remove(histo_file)

    return spectrum

## Overlay of the k-mer spectra

To compare the different values of $k$ directly, we draw all the spectra on the same axes.

Left: raw multiplicity. The single-copy peak moves to the left as $k$ grows, because $\lambda_{\mathrm{true}}(k)$ decreases (fewer error-free k-mers, fewer starting positions).

Right: multiplicity divided by the empirical single-copy peak, and counts divided by the number of distinct k-mers. In this normalized view the spectra become comparable: the height of the error component on the left shows how much the sequencing errors weigh for each $k$, and the right tail shows the repeated k-mers.

In [ ]:
# >>> NUOVA CELLA
# ============================================================
# OVERLAY OF THE SPECTRA FOR ALL k
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for _, row in results_df.iterrows():

    k = int(row["k"])
    genomic_peak = int(row["genomic_peak"])
    distinct = row["distinct_kmers"]

    spectrum = spectra[k]

    max_x = int(
        min(
            max(spectrum.keys()),
            max(200, 2.5 * genomic_peak)
        )
    )

    x = np.arange(1, max_x + 1)

    y = np.array([spectrum.get(m, 0) for m in x], dtype=float)

    keep = y > 0

    axes[0].plot(
        x[keep],
        y[keep],
        label=f"k = {k}"
    )

    axes[1].plot(
        x[keep] / genomic_peak,
        y[keep] / distinct,
        label=f"k = {k}"
    )

axes[0].set_yscale("log")
axes[0].set_xlabel("k-mer multiplicity")
axes[0].set_ylabel("Number of distinct k-mers (log scale)")
axes[0].set_title("Raw spectra")

axes[1].set_yscale("log")
axes[1].set_xlabel("multiplicity / single-copy peak")
axes[1].set_ylabel("Fraction of distinct k-mers (log scale)")
axes[1].set_title("Normalized spectra")

for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## Comparison across different values of k

The theoretical lower bound only excludes values that are clearly too short.

The actual selection of $k$ is based on the empirical spectra.

For each candidate value, we estimate:

$$
f_{\mathrm{error}}(k),
$$

the fraction of distinct k-mers likely generated by sequencing errors, and

$$
f_{\mathrm{repeat}}(k),
$$

the fraction of solid k-mers whose multiplicity suggests that they occur more
than once in the genome.

Increasing $k$ should reduce ambiguity caused by repeated genomic sequences,
but longer k-mers are also more likely to contain sequencing errors.

We therefore compare these two empirical quantities rather than deriving an
"optimal" $k$ from a theoretical formula.

In [ ]:
# ============================================================
# EMPIRICAL TRADE-OFF
# ============================================================

plt.figure(
    figsize=(9, 5)
)

plt.plot(
    results_df["k"],
    results_df[
        "erroneous_type_fraction"
    ],
    marker="o",
    label="Likely erroneous k-mers"
)

plt.plot(
    results_df["k"],
    results_df[
        "repeated_fraction_among_solid"
    ],
    marker="o",
    label="Likely repeated solid k-mers"
)

plt.xlabel(
    "k"
)

plt.ylabel(
    "Fraction of distinct k-mers"
)

plt.title(
    "Empirical trade-off for the choice of k"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.show()

In [ ]:
# ============================================================
# SUMMARY TABLE
# ============================================================

summary_df = results_df[[
    "k",
    "genomic_peak",
    "threshold",
    "repeat_threshold",
    "erroneous_type_fraction",
    "repeated_fraction_among_solid"
]].copy()

display(
    summary_df.round(4)
)

In [ ]:
# ============================================================
# PARETO-EFFICIENT CANDIDATES
# ============================================================

pareto_candidates = []

for i, row_i in results_df.iterrows():

    err_i = row_i[
        "erroneous_type_fraction"
    ]

    rep_i = row_i[
        "repeated_fraction_among_solid"
    ]

    dominated = False

    for j, row_j in results_df.iterrows():

        if i == j:
            continue

        err_j = row_j[
            "erroneous_type_fraction"
        ]

        rep_j = row_j[
            "repeated_fraction_among_solid"
        ]

        if (
            err_j <= err_i
            and rep_j <= rep_i
            and (
                err_j < err_i
                or rep_j < rep_i
            )
        ):

            dominated = True
            break

    if not dominated:

        pareto_candidates.append(
            int(row_i["k"])
        )


print(
    "Pareto-efficient candidate k values:",
    pareto_candidates
)

## Linearity of the filtered de Bruijn graph

After identifying a filtering threshold for each value of $k$, we keep only
solid k-mers, i.e. k-mers satisfying $c(w) \geq t_k$.

The de Bruijn graph is then constructed using only these trusted k-mers. Each
k-mer defines an edge between its $(k-1)$-mer prefix and its $(k-1)$-mer
suffix.

For a perfectly linear genomic region, an internal node should have one incoming
and one outgoing edge. Branching nodes satisfy

$$
\operatorname{indegree}(v)>1
\quad\text{or}\quad
\operatorname{outdegree}(v)>1.
$$

For each candidate $k$, we therefore compute the fraction of branching nodes

$$
f_{\mathrm{branch}}(k)
=
\frac{\#\text{ branching nodes}}
{\#\text{ nodes}}.
$$

A smaller branching fraction indicates a graph closer to a collection of linear
paths. The graph analysis is performed only after removing low-frequency,
non-trusted k-mers, so that sequencing errors do not artificially create
branches.

In [ ]:
# ============================================================
# FILTERED DE BRUIJN GRAPH
# ============================================================

def build_filtered_debruijn_masks(
    reads,
    k,
    canonical_counts,
    threshold
):

    # For each node we only store which bases can follow it
    # or precede it.
    #
    # Each of the 4 bits corresponds to A, C, G, T.
    out_mask = {}
    in_mask = {}

    kmer_mask = (1 << (2 * k)) - 1
    reverse_shift = 2 * (k - 1)

    # Mask for a (k-1)-mer
    node_mask = (1 << (2 * (k - 1))) - 1

    solid_occurrences = 0

    for read in reads:

        forward = 0
        reverse = 0
        valid_length = 0

        for base in read:

            if base not in BASE_TO_BITS:

                forward = 0
                reverse = 0
                valid_length = 0
                continue

            b = BASE_TO_BITS[base]

            # Update forward k-mer
            forward = (
                ((forward << 2) | b)
                & kmer_mask
            )

            # Update reverse complement
            complement = 3 - b

            reverse = (
                (reverse >> 2)
                |
                (complement << reverse_shift)
            )

            valid_length += 1

            if valid_length < k:
                continue

            # Canonical representation is used ONLY
            # to check whether the k-mer is trusted
            canonical = min(
                forward,
                reverse
            )

            if canonical_counts.get(canonical, 0) < threshold:
                continue

            solid_occurrences += 1

            # ------------------------------------------------
            # Directed edge in the orientation of the read
            #
            # k-mer: b1 b2 ... bk
            #
            # prefix = b1 ... b(k-1)
            # suffix = b2 ... bk
            # ------------------------------------------------

            prefix = forward >> 2

            suffix = (
                forward
                & node_mask
            )

            first_base = (
                forward
                >> (2 * (k - 1))
            ) & 3

            last_base = (
                forward
                & 3
            )

            # Record the possible outgoing base
            out_mask[prefix] = (
                out_mask.get(prefix, 0)
                | (1 << last_base)
            )

            # Record the possible incoming base
            in_mask[suffix] = (
                in_mask.get(suffix, 0)
                | (1 << first_base)
            )

    return (
        out_mask,
        in_mask,
        solid_occurrences
    )

In [ ]:
# ============================================================
# GRAPH LINEARITY METRICS
# ============================================================

def compute_graph_metrics(
    out_mask,
    in_mask
):

    nodes = (
        set(out_mask.keys())
        |
        set(in_mask.keys())
    )

    n_nodes = len(nodes)

    n_edges = sum(
        mask.bit_count()
        for mask in out_mask.values()
    )

    branching_nodes = 0
    linear_nodes = 0
    terminal_nodes = 0

    for node in nodes:

        indegree = (
            in_mask.get(node, 0)
            .bit_count()
        )

        outdegree = (
            out_mask.get(node, 0)
            .bit_count()
        )

        # Branch produced by ambiguity
        if (
            indegree > 1
            or outdegree > 1
        ):
            branching_nodes += 1

        # Internal node belonging to a linear path
        if (
            indegree == 1
            and outdegree == 1
        ):
            linear_nodes += 1

        # Beginning/end of a path
        if (
            indegree == 0
            or outdegree == 0
        ):
            terminal_nodes += 1

    if n_nodes > 0:

        branching_fraction = (
            branching_nodes
            / n_nodes
        )

        linear_fraction = (
            linear_nodes
            / n_nodes
        )

        terminal_fraction = (
            terminal_nodes
            / n_nodes
        )

    else:

        branching_fraction = np.nan
        linear_fraction = np.nan
        terminal_fraction = np.nan

    return {
        "nodes": n_nodes,
        "edges": n_edges,
        "branching_nodes": branching_nodes,
        "branching_fraction": branching_fraction,
        "linear_fraction": linear_fraction,
        "terminal_fraction": terminal_fraction
    }

## Unitigs: how many contiguous sequences does the filtered graph give?

The branching fraction tells us how many nodes are ambiguous, but not how long the linear paths are. A more direct measure is to extract the **unitigs**, i.e. the maximal non-branching paths of the filtered de Bruijn graph.

A unitig starts and ends at a node that is not "1 in, 1 out", and all its internal nodes have $\operatorname{indegree}=\operatorname{outdegree}=1$. A unitig made of $e$ edges (k-mers) spans $e+k-1$ bases.

For each $k$ we compute the number of unitigs, the longest one and the **N50** (the length $x$ such that unitigs of length $\geq x$ cover at least half of the total unitig length).

Small $k$: repeats create branches and the unitigs are cut. Large $k$: the sequencing errors and the lower multiplicity $\lambda_{\mathrm{true}}(k)$ create holes in the graph and the unitigs are cut again. A good $k$ should therefore give a large N50.

Since the graph is built from reads of both strands, every unitig appears together with its reverse complement. N50 is not affected, while the total length is about twice the genome length covered.

In [ ]:
# >>> NUOVA CELLA
# ============================================================
# UNITIGS OF THE FILTERED DE BRUIJN GRAPH
# ============================================================

def n50_from_lengths(lengths):

    if len(lengths) == 0:
        return 0

    arr = np.sort(
        np.asarray(lengths)
    )[::-1]

    cumulative = np.cumsum(arr)

    index = np.searchsorted(
        cumulative,
        cumulative[-1] / 2
    )

    return int(arr[index])


def compute_unitig_stats(
    out_mask,
    in_mask,
    k
):

    node_mask = (1 << (2 * (k - 1))) - 1

    nodes = (
        set(out_mask.keys())
        |
        set(in_mask.keys())
    )

    unitig_lengths = []

    for node in nodes:

        indegree = in_mask.get(node, 0).bit_count()
        outdegree = out_mask.get(node, 0).bit_count()

        # A unitig starts from a node which is NOT 1-in-1-out
        if indegree == 1 and outdegree == 1:
            continue

        mask_out = out_mask.get(node, 0)

        for base in range(4):

            if not (mask_out >> base) & 1:
                continue

            n_edges = 1

            current = (
                ((node << 2) | base)
                & node_mask
            )

            # Follow the path while it does not branch
            while (
                in_mask.get(current, 0).bit_count() == 1
                and
                out_mask.get(current, 0).bit_count() == 1
            ):

                next_base = (
                    out_mask[current].bit_length() - 1
                )

                current = (
                    ((current << 2) | next_base)
                    & node_mask
                )

                n_edges += 1

            # e edges = e k-mers -> e + k - 1 bases
            unitig_lengths.append(
                n_edges + k - 1
            )

    if len(unitig_lengths) == 0:

        return {
            "n_unitigs": 0,
            "longest_unitig": 0,
            "n50_unitigs": 0,
            "total_unitig_length": 0
        }

    return {
        "n_unitigs": len(unitig_lengths),
        "longest_unitig": max(unitig_lengths),
        "n50_unitigs": n50_from_lengths(unitig_lengths),
        "total_unitig_length": int(sum(unitig_lengths))
    }


In [ ]:
# >>> CELLA MODIFICATA (al posto della vecchia cella 25)
# ============================================================
# GRAPH LINEARITY AND UNITIGS FOR EACH CANDIDATE k
# ============================================================

graph_results = []

for k in K_VALUES:

    print()
    print("=" * 55)
    print("Filtered de Bruijn graph for k =", k)
    print("=" * 55)

    # Threshold already estimated from the spectrum
    threshold = int(
        results_df.loc[
            results_df["k"] == k,
            "threshold"
        ].iloc[0]
    )

    print(
        "Trusted k-mer threshold:",
        threshold
    )

    # --------------------------------------------------------
    # Recompute canonical counts
    #
    # They were deleted in the previous loop to save memory.
    # --------------------------------------------------------

    canonical_counts = (
        count_canonical_kmers(
            reads,
            k
        )
    )

    # --------------------------------------------------------
    # Build graph using ONLY trusted k-mers
    # --------------------------------------------------------

    out_mask, in_mask, solid_occurrences = (
        build_filtered_debruijn_masks(
            reads=reads,
            k=k,
            canonical_counts=canonical_counts,
            threshold=threshold
        )
    )

    # --------------------------------------------------------
    # Graph statistics
    # --------------------------------------------------------

    metrics = compute_graph_metrics(
        out_mask,
        in_mask
    )

    # NEW: unitig statistics
    unitigs = compute_unitig_stats(
        out_mask,
        in_mask,
        k
    )

    graph_results.append({
        "k":
            k,

        "threshold":
            threshold,

        "solid_occurrences":
            solid_occurrences,

        "nodes":
            metrics["nodes"],

        "edges":
            metrics["edges"],

        "branching_nodes":
            metrics["branching_nodes"],

        "branching_fraction":
            metrics["branching_fraction"],

        "linear_fraction":
            metrics["linear_fraction"],

        "terminal_fraction":
            metrics["terminal_fraction"],

        "n_unitigs":
            unitigs["n_unitigs"],

        "longest_unitig":
            unitigs["longest_unitig"],

        "n50_unitigs":
            unitigs["n50_unitigs"],

        "total_unitig_length":
            unitigs["total_unitig_length"]
    })

    print(
        "Number of nodes:",
        metrics["nodes"]
    )

    print(
        "Number of edges:",
        metrics["edges"]
    )

    print(
        "Branching fraction:",
        round(
            metrics["branching_fraction"],
            6
        )
    )

    print(
        "Linear fraction:",
        round(
            metrics["linear_fraction"],
            6
        )
    )

    print(
        "Number of unitigs:",
        unitigs["n_unitigs"]
    )

    print(
        "Longest unitig:",
        unitigs["longest_unitig"]
    )

    print(
        "N50 of the unitigs:",
        unitigs["n50_unitigs"]
    )

    # IMPORTANT:
    # free memory before moving to the next k

    del canonical_counts
    del out_mask
    del in_mask

    gc.collect()


graph_df = pd.DataFrame(
    graph_results
)

display(
    graph_df.round(6)
)


In [ ]:
# ============================================================
# GRAPH LINEARITY ACROSS k
# ============================================================

plt.figure(figsize=(9, 5))

plt.plot(
    graph_df["k"],
    graph_df["branching_fraction"],
    marker="o",
    label="Branching fraction"
)

plt.plot(
    graph_df["k"],
    graph_df["linear_fraction"],
    marker="o",
    label="Linear-node fraction"
)

plt.xlabel("k")
plt.ylabel("Fraction of graph nodes")

plt.title(
    "Linearity of the filtered de Bruijn graph"
)

plt.legend()

plt.grid(alpha=0.3)

plt.show()

In [ ]:
# >>> NUOVA CELLA
# ============================================================
# UNITIG CONTIGUITY ACROSS k
# ============================================================

fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(
    graph_df["k"],
    graph_df["n50_unitigs"],
    marker="o",
    label="N50 of the unitigs"
)

ax1.plot(
    graph_df["k"],
    graph_df["longest_unitig"],
    marker="s",
    linestyle="--",
    label="Longest unitig"
)

ax1.set_xlabel("k")
ax1.set_ylabel("Length (bases)")
ax1.set_yscale("log")
ax1.grid(alpha=0.3)

ax1.set_title(
    "Contiguity of the filtered de Bruijn graph"
)

ax1.legend()

plt.show()


In [ ]:
# >>> CELLA MODIFICATA (al posto della vecchia cella 27)
# ============================================================
# FINAL COMPARISON FOR k
# ============================================================

k_comparison = (
    results_df[[
        "k",
        "threshold",
        "erroneous_type_fraction",
        "erroneous_occurrence_fraction",
        "repeated_fraction_among_solid"
    ]]
    .merge(
        graph_df[[
            "k",
            "branching_fraction",
            "linear_fraction",
            "n_unitigs",
            "n50_unitigs",
            "longest_unitig"
        ]],
        on="k"
    )
)

display(
    k_comparison.round(5)
)

best_row = k_comparison.loc[
    k_comparison["n50_unitigs"].idxmax()
]

print()
print(
    "k with the largest unitig N50:",
    int(best_row["k"])
)
print(
    "N50:",
    int(best_row["n50_unitigs"])
)
print(
    "Branching fraction:",
    round(best_row["branching_fraction"], 5)
)
print(
    "Likely erroneous k-mer fraction:",
    round(best_row["erroneous_type_fraction"], 5)
)
